# 3.0 True 7-Day Recursive Backtest + Naive Baselines（真实 7 天递归回测 + 朴素基线）

目标（对应论文/报告里的“诚实披露”）：

1. **单步预测（15-min ahead）测试集 MAE** —— 与 `2.0_forecast_visualization.ipynb` Section 1 同口径，用于复现基线（LightGBM V3.1 ≈ 2.6390、XGBoost V4 ≈ 2.7020）。
2. **真正的 7 天（168 h = 672 个 15-min）递归预测** —— 在保留测试集上连续递归预测，**预测窗口内绝不喂真实电价**：第 1 步只用窗口开始前的真实历史；之后每步把上一步的预测值喂回价格滞后/滚动特征。得到“真实 MAE / RMSE”。
3. **Naive 持久化基线**：`Naive_24h`（24 h 前同 slot = shift(96)）与 `Naive_7d`（7 天前同 slot = shift(672)）。表格中在 V1 之上新增 **Naive Baseline (Yesterday's Price)** 行；7 天递归的对手是 **Naive_7d**（整窗无泄漏）。

关键口径（重要）：
- 只用本地模型 `models/saved/lightgbm_v3_1.pkl` 与 `models/saved/xgboost_v4.pkl`（就是 README 表里 2.6390 / 2.7020 那两个），**不是**线上 `predictions/*.csv` 的数字。
- 价格类特征按训练 notebook 公式重建：`price.shift(k)`、`price.shift(1).rolling(W)`（`std` 用 pandas 默认 `ddof=1`）。
- 外生特征（天气 / 电网 / 核电等）在窗口内取 CSV 的**已实现值**（= 完美天气预报，与单步评估完全一致），从而让“单步 vs 7 天递归”的差距**只来自价格自回归误差累积**。
- `src/features.py` 的 `build_features()` 与训练表存在若干细微 train/serve 偏差（`season` 映射、rolling-std 的 `ddof`、小时内天气取整、电网对齐等），故本回测不用它重建，而是严格复刻训练公式，见附录 A。


## 0. Setup（数据 + 时序 80/20 切分）

In [1]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# notebook 位于 data_visualization/；向上定位仓库根
ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (ROOT.parent / 'data').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))   # （可选）如需复用 src/features.py

HELSINKI = 'Europe/Helsinki'
DATA_PATH = ROOT / 'data' / 'convertData' / 'V3.1_15min_features.csv'
MODELS_DIR = ROOT / 'models' / 'saved'
MODEL_NAMES = ['lightgbm_v3_1', 'xgboost_v4']   # 两个最新模型

H = 4                 # 每小时 15-min 步数
D = 24 * H            # 96 步 = 24 h
W7 = 7 * D            # 672 步 = 7 天
FORECAST_STEPS = W7   # 7 天递归 = 672 个 15-min 步

# ── 加载共享 V3.1 特征表，统一时间轴（与 2.0 评估 notebook 一致）─────────────
df = pd.read_csv(DATA_PATH)
df['datetime'] = pd.to_datetime(df['datetime'], utc=True).dt.tz_convert(HELSINKI)
df = df.sort_values('datetime').reset_index(drop=True)
print('rows:', len(df), '| range:', df['datetime'].iloc[0], '->', df['datetime'].iloc[-1])

n = len(df)
train_end = n - int(n * 0.20)                 # 时序 80/20：测试集 = 最后 20%
y = df['price'].to_numpy()
test_pos = np.arange(train_end, n)
print('train_end =', train_end, '| test rows =', len(test_pos))
print('test period:', df['datetime'].iloc[train_end], '->', df['datetime'].iloc[-1])


rows: 105216 | range: 2023-01-01 00:00:00+02:00 -> 2025-12-31 23:45:00+02:00
train_end = 84173 | test rows = 21043
test period: 2025-05-26 20:15:00+03:00 -> 2025-12-31 23:45:00+02:00


## 1. 单步（15-min ahead）测试集评估（复现 README 基线）

每个测试样本直接用 CSV 里的滞后特征（上一步是真实价）预测下一步 —— 这就是表格里
`Test MAE` 那一列的含义（LightGBM V3.1 ≈ 2.6390、XGBoost V4 ≈ 2.7020）。


In [2]:
X = df.drop(columns=['datetime', 'price'])
yt = y[train_end:]
single = {}   # name -> {'MAE','RMSE','R2'}
for name in MODEL_NAMES:
    meta = joblib.load(MODELS_DIR / f'{name}.pkl')
    cols = list(meta['feature_cols'])
    yp = meta['model'].predict(X.iloc[train_end:][cols])
    single[name] = {
        'MAE':  mean_absolute_error(yt, yp),
        'RMSE': float(np.sqrt(mean_squared_error(yt, yp))),
        'R2':   r2_score(yt, yp),
    }
    print(f'{name}: single-step(15-min ahead) MAE={single[name]["MAE"]:.4f} | '
          f'RMSE={single[name]["RMSE"]:.4f} | R2={single[name]["R2"]:.4f}')


lightgbm_v3_1: single-step(15-min ahead) MAE=2.6390 | RMSE=7.8957 | R2=0.9740
xgboost_v4: single-step(15-min ahead) MAE=2.7020 | RMSE=8.0376 | R2=0.9730


### 1.1 Naive 基线（单步口径，供表格在 V1 之上加行）

- `Naive_24h`：直接复制 **24 h 前同 slot** 的真实价（= “昨天同一时刻”）作为今天该 slot 的预测 → 表格行 **Naive Baseline (Yesterday's Price)**。
- `Naive_7d`：直接复制 **7 天前同 slot** 的真实价（= “上周同一时刻”）。

两者都在**同一测试集**上算 MAE/RMSE/R²，与模型行可比。


In [3]:
price = df['price']
naive24 = price.shift(D).to_numpy()    # 96 步前
naive7d = price.shift(W7).to_numpy()   # 672 步前

for key, pred in [('Naive_24h', naive24), ('Naive_7d', naive7d)]:
    yp = pred[train_end:]
    single[key] = {
        'MAE':  mean_absolute_error(yt, yp),
        'RMSE': float(np.sqrt(mean_squared_error(yt, yp))),
        'R2':   r2_score(yt, yp),
    }
    print(f'{key}: MAE={single[key]["MAE"]:.4f} | RMSE={single[key]["RMSE"]:.4f} | '
          f'R2={single[key]["R2"]:.4f}  (same test set)')

print('\nSingle-step leaderboard (MAE/RMSE lower = better):')
print(pd.DataFrame(single).T.round(4).to_string())


Naive_24h: MAE=31.7224 | RMSE=51.1850 | R2=-0.0932  (same test set)
Naive_7d: MAE=40.2503 | RMSE=61.7619 | R2=-0.5917  (same test set)

Single-step leaderboard (MAE/RMSE lower = better):
                   MAE     RMSE      R2
lightgbm_v3_1   2.6390   7.8957  0.9740
xgboost_v4      2.7020   8.0376  0.9730
Naive_24h      31.7224  51.1850 -0.0932
Naive_7d       40.2503  61.7619 -0.5917


## 2. 真正的 7 天（168 h）递归预测回测

对测试集内每个 7 天窗口 `[s, s+672)`：

- **价格历史**：只含 `s` 之前的真实价 + 窗口内已产生的预测值（**不喂窗口内真实电价**）。
- 第 1 步：用 `s` 之前的真实历史构造滞后/滚动特征 → 预测 `s`；
  第 2 步：把 `s` 的预测当作历史 → 预测 `s+15min`；……直到填满 672 步。
- 价格特征用下面的公式（与训练 notebook 完全一致）重建；外生特征取 CSV 已实现值。

窗口：从测试集起点 `train_end` 起每 7 天一个不重叠窗口（约 31 个），覆盖整个测试集。


In [15]:
# ── 价格类特征：严格复刻训练公式 ────────────────────────────────────────────
# price_lag_k = price.shift(k)；price_rolling_* = price.shift(1).rolling(W).{mean,std,min,max}
# （pandas rolling.std 默认 ddof=1，故用 np.std(..., ddof=1)）
PRICE_LAG = {'price_lag_1': 1, 'price_lag_2': 2, 'price_lag_4': 4, 'price_lag_8': 8,
             'price_lag_16': 16, 'price_lag_32': 32, 'price_lag_96': 96, 'price_lag_672': 672}
PRICE_ROLL = {'price_rolling_mean_1h': (4, 'mean'), 'price_rolling_std_1h': (4, 'std'),
              'price_rolling_mean_6h': (24, 'mean'), 'price_rolling_mean_24h': (96, 'mean'),
              'price_rolling_std_24h': (96, 'std'), 'price_rolling_min_24h': (96, 'min'),
              'price_rolling_max_24h': (96, 'max'), 'price_rolling_mean_7d': (672, 'mean')}
PRICE_COLS = list(PRICE_LAG) + list(PRICE_ROLL)


def recursive_window(model, cols, exog, pos_s):
    """从 pos_s 递归预测 FORECAST_STEPS(672) 步，不喂窗口内真实电价。

    exog: 全量特征表（CSV 已实现值）；外生列按需从 exog 取，价格列按公式重建。
    返回 (preds, actuals, pos_arr)。
    """
    exog_cols = [c for c in cols if c not in PRICE_COLS]
    exog_idx = {c: i for i, c in enumerate(exog_cols)}
    exog_arr = exog[exog_cols].to_numpy() if exog_cols else np.zeros((len(exog), 0))
    eff = y[:pos_s].tolist()          # 已知历史真实价；预测后逐个追加（不追加真实价）
    nw = FORECAST_STEPS
    preds = np.empty(nw)
    Xrow = np.empty(len(cols))
    for i in range(nw):
        p = pos_s + i
        for j, c in enumerate(cols):
            if c in exog_idx:
                Xrow[j] = exog_arr[p, exog_idx[c]]
            elif c in PRICE_LAG:
                Xrow[j] = eff[p - PRICE_LAG[c]]
            else:                     # rolling
                W, op = PRICE_ROLL[c]
                seg = eff[p - W:p]
                if op == 'mean':
                    v = float(np.mean(seg))
                elif op == 'std':
                    v = float(np.std(seg, ddof=1))
                elif op == 'min':
                    v = float(min(seg))
                else:
                    v = float(max(seg))
                Xrow[j] = v
        # 带列名的 DataFrame，避免 “no valid feature names” 警告，且与生产 predict_system 一致
        Xf = pd.DataFrame(Xrow.reshape(1, -1), columns=cols)
        preds[i] = float(model.predict(Xf)[0])
        eff.append(preds[i])
    actuals = y[pos_s:pos_s + nw].copy()
    pos_arr = np.arange(pos_s, pos_s + nw)
    return preds, actuals, pos_arr


In [7]:
# ── 外生特征准备：天气列在 2025 测试区有零星缺失（temp ~15 / wind ~12 行，约 0.2%）。
#    保留 CSV 原有列（不做公式重算，避免与训练定义有出入），仅对整表做 ffill+bfill
#    缺口填充（缺失值沿用邻近观测，相当于“天气预报可用”）。
#    价格派生列前 672 行的 NaN 不属于外生特征（PRICE_COLS 在递归中重建），不受影响。
Xall = df.drop(columns=['datetime', 'price']).ffill().bfill()
print('gap-filled exogenous Xall cols =', Xall.shape[1])


gap-filled exogenous Xall cols = 68


### 2.1 健全性检查

1. 价格公式重建 vs CSV 预计算列：应精确一致（< 1e-6）。
2. 测试区外生特征无缺失。


In [8]:
# 2) 测试区外生特征无缺失（Xall 为前面清洗好的外生矩阵）
for name in MODEL_NAMES:
    meta = joblib.load(MODELS_DIR / f'{name}.pkl')
    cols = list(meta['feature_cols'])
    exog_cols = [c for c in cols if c not in PRICE_COLS]
    nan_cnt = int(Xall.iloc[train_end:][exog_cols].isna().sum().sum())
    print(f'{name}: exog cols = {len(exog_cols)} | NaN in test region = {nan_cnt}')
    assert nan_cnt == 0, 'exogenous features have NaN in the test region!'


lightgbm_v3_1: exog cols = 52 | NaN in test region = 0
xgboost_v4: exog cols = 52 | NaN in test region = 0


### 2.2 跑全部窗口（约 31 个 7 天窗口 × 2 个模型，需要几分钟）

In [16]:
WINDOW_STEP = W7   # 不重叠：每 7 天一个窗口
starts = list(range(train_end, n - FORECAST_STEPS + 1, WINDOW_STEP))
print('num 7-day windows:', len(starts), '| first start:', df['datetime'].iloc[starts[0]],
      '| last start:', df['datetime'].iloc[starts[-1]])

results = {}   # name -> dict(preds, actuals, pos, naive7)
for name in MODEL_NAMES:
    meta = joblib.load(MODELS_DIR / f'{name}.pkl')
    model, cols = meta['model'], list(meta['feature_cols'])
    t0 = time.time()
    preds_l, act_l, pos_l = [], [], []
    for pos_s in starts:
        pr, ac, ps = recursive_window(model, cols, Xall, pos_s)
        preds_l.append(pr); act_l.append(ac); pos_l.append(ps)
    preds = np.concatenate(preds_l)
    actuals = np.concatenate(act_l)
    pos = np.concatenate(pos_l)
    naive7 = y[pos - W7]                 # 上周同 slot —— 全在窗口外，无泄漏
    results[name] = dict(preds=preds, actuals=actuals, pos=pos, naive7=naive7)
    print(f'{name}: done in {time.time() - t0:.1f}s | samples = {len(preds):,}')


num 7-day windows: 31 | first start: 2025-05-26 20:15:00+03:00 | last start: 2025-12-22 19:15:00+02:00
lightgbm_v3_1: done in 31.4s | samples = 20,832
xgboost_v4: done in 117.7s | samples = 20,832


## 3. 汇总结果与摘要句

In [17]:
def _mae(a, b): return float(mean_absolute_error(a, b))
def _rmse(a, b): return float(np.sqrt(mean_squared_error(a, b)))

rec_rows = []
for name in MODEL_NAMES:
    r = results[name]
    rec_rows.append({
        'model':          name,
        'single_MAE':     single[name]['MAE'],            # 15-min ahead（单步）
        'recursive_MAE':  _mae(r['actuals'], r['preds']), # 7 天递归
        'recursive_RMSE': _rmse(r['actuals'], r['preds']),
        'Naive7d_MAE':    _mae(r['actuals'], r['naive7']),
        'Naive7d_RMSE':   _rmse(r['actuals'], r['naive7']),
        'beats_Naive7d':  _mae(r['actuals'], r['preds']) < _mae(r['actuals'], r['naive7']),
    })
rec_tab = pd.DataFrame(rec_rows)
print('=== 单步 vs 7 天递归 vs Naive_7d（同一批测试样本） ===')
print(rec_tab.round(4).to_string(index=False))


=== 单步 vs 7 天递归 vs Naive_7d（同一批测试样本） ===
        model  single_MAE  recursive_MAE  recursive_RMSE  Naive7d_MAE  Naive7d_RMSE  beats_Naive7d
lightgbm_v3_1       2.639        37.2911         59.7681      40.0616       61.6528           True
   xgboost_v4       2.702        32.1243         46.0824      40.0616       61.6528           True


In [18]:
print('\n' + '=' * 78)
for _, row in rec_tab.iterrows():
    print(f'[{row["model"]}]')
    print(f'  单步预测（15-min ahead，测试集）MAE          = {row["single_MAE"]:.4f}')
    print(f'  7 天递归预测（168 h，不喂真实电价）MAE = {row["recursive_MAE"]:.4f} | RMSE = {row["recursive_RMSE"]:.4f}')
    print(f'  同窗口 Naive_7d（上周同 slot）MAE        = {row["Naive7d_MAE"]:.4f} | RMSE = {row["Naive7d_RMSE"]:.4f}')
    print(f'  7 天递归是否跑赢 Naive_7d：{"YES ✅" if row["beats_Naive7d"] else "NO ❌"}')
print('=' * 78)

# 论文/摘要可直接引用的切分句（取两者中 7 天递归 MAE 更低者作主句，数值为真实计算结果）
best = rec_tab.sort_values('recursive_MAE').iloc[0]
print(f'SUMMARY: 单步预测（15-min ahead）达到了 {best["single_MAE"]:.2f} MAE，'
      f'而 7 天递归预测的 MAE 为 {best["recursive_MAE"]:.2f}'
      f'（RMSE {best["recursive_RMSE"]:.2f}，对应 {best["model"]}）。')



[lightgbm_v3_1]
  单步预测（15-min ahead，测试集）MAE          = 2.6390
  7 天递归预测（168 h，不喂真实电价）MAE = 37.2911 | RMSE = 59.7681
  同窗口 Naive_7d（上周同 slot）MAE        = 40.0616 | RMSE = 61.6528
  7 天递归是否跑赢 Naive_7d：YES ✅
[xgboost_v4]
  单步预测（15-min ahead，测试集）MAE          = 2.7020
  7 天递归预测（168 h，不喂真实电价）MAE = 32.1243 | RMSE = 46.0824
  同窗口 Naive_7d（上周同 slot）MAE        = 40.0616 | RMSE = 61.6528
  7 天递归是否跑赢 Naive_7d：YES ✅
SUMMARY: 单步预测（15-min ahead）达到了 2.70 MAE，而 7 天递归预测的 MAE 为 32.12（RMSE 46.08，对应 xgboost_v4）。


### 3.1 可选：按 lead day 展开（误差如何在 7 天内累积）

In [19]:
for name in MODEL_NAMES:
    r = results[name]
    within = (r['pos'] - train_end) % W7     # 在每个 7 天窗口内的偏移（0..671）
    lead_day = within // D + 1               # 窗口内第 1..7 天
    per = [_mae(r['actuals'][lead_day == d], r['preds'][lead_day == d]) for d in range(1, 8)]
    sizes = [int((lead_day == d).sum()) for d in range(1, 8)]
    print(f'{name}: recursive MAE by lead day 1..7 -> '
          + ' | '.join(f'day{d}={v:.2f}({s})' for d, v, s in zip(range(1, 8), per, sizes)))


lightgbm_v3_1: recursive MAE by lead day 1..7 -> day1=30.89(2976) | day2=41.09(2976) | day3=42.37(2976) | day4=47.94(2976) | day5=34.10(2976) | day6=28.42(2976) | day7=36.24(2976)
xgboost_v4: recursive MAE by lead day 1..7 -> day1=31.99(2976) | day2=36.79(2976) | day3=36.14(2976) | day4=39.20(2976) | day5=28.05(2976) | day6=23.20(2976) | day7=29.48(2976)


## 附录 A. 为什么不用 src/features.py 重建特征（train/serve 偏差）

`src/features.py` 的 `build_features()` 是线上在用的生产代码，但逐特征对照训练表发现 16 处偏差：

- `season`：`src/features.py` 把 6 月映射为 2，训练表为 3（4 季分组不同）。
- `price_rolling_std_*`：`src/features.py` 用 `np.std`（ddof=0），训练用 pandas `rolling.std`（ddof=1）。
- 电网走廊（`fi_ee`/`fi_se_*`）当前值与训练表错位；小时内天气被 `WeatherBuffer` 取整到整点。

这些偏差意味着“线上递归成绩”（`predictions/*.csv`）额外混入了 train/serve 漂移。为了把
“单步 vs 7 天递归”的差距**纯粹**归因于价格自回归误差累积，本 notebook 严格复刻训练公式
重建价格特征、外生特征直接取训练表已实现值——这也正是 2.0 评估 notebook 单步评估的同一特征口径。
